# Preprocessing 02 — Emotion Re-labelling + Image Captions

**What this notebook does:**
- Loads CLIP image embeddings already computed in `Preprocessing01` (no ViT re-run)
- Re-scores each image against the 6 target emotion classes using the saved embeddings
- Generates a natural-language description (caption) for each image with **BLIP-2**
- Encodes captions into sentence embeddings with **SBERT**
- Saves an enriched parquet + updated `.npy` files

**Depends on (from Preprocessing01):**
```
lamem_features3/
  clip_embeddings.npy          # (N, 768) — CLIP ViT-L/14 image features
  lamem_features_full.parquet  # metadata (name, memscore, local_path, ...)
lamem_images/                  # images already downloaded locally
```

In [1]:
!pip install -q git+https://github.com/openai/CLIP.git transformers accelerate
!pip install -q sentence-transformers

In [2]:
pip install fastparquet 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import torch

import clip
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from sentence_transformers import SentenceTransformer


from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

/home/gabriella/Documents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# ROOT_DIR = "/content/drive/MyDrive/CompNeuroscience-P1"

ROOT_DIR = "/home/gabriella/Documents/MemorabilityEmotions"

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

PREV_OUTPUT_DIR = Path(ROOT_DIR) / "lamem_features"
IMAGES_DIR      = Path(ROOT_DIR) / "lamem_images"

CLIP_EMB_PATH   = PREV_OUTPUT_DIR / "clip_embeddings.npy"
PREV_PARQUET    = Path(ROOT_DIR) / "lamem_features" / "lamem_features_full.parquet"

OUTPUT_DIR = Path(ROOT_DIR) / "lamem_features_v3"
OUTPUT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 500

EMOTIONS = [
    "happiness",
    "sadness",
    "fear",
    "anger",
    "disgust",
    "surprise",
]

Device: cuda


## 1 — Load existing metadata + CLIP embeddings (no ViT re-run)

In [11]:
PREV_PARQUET

PosixPath('/home/gabriella/Documents/MemorabilityEmotions/lamem_features/lamem_features_full.parquet')

In [12]:
base_df = pd.read_parquet(PREV_PARQUET)
print(f"Loaded metadata: {base_df.shape}")
print(base_df.columns.tolist())

Loaded metadata: (16815, 14)
['name', 'local_path', 'dataset', 'memscore', 'original_emotion', 'clip_emotion', 'clip_emotion_conf', 'caption', 'error', 'emotion_amusement', 'emotion_excitement', 'emotion_awe', 'emotion_contentment', 'emotion_sadness']


In [13]:
clip_matrix = np.load(CLIP_EMB_PATH)   # (N, 768)
print(f"CLIP embeddings shape: {clip_matrix.shape}")
assert len(clip_matrix) == len(base_df), "Mismatch between parquet rows and embedding rows!"

CLIP embeddings shape: (16815, 768)


## 2 — Re-score emotions with the 6-class taxonomy

We only need the **text encoder** from CLIP — the image embeddings already exist.

In [14]:
clip_model, _ = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
print("CLIP text encoder ready.")

100%|████████████████████████████████████████| 890M/890M [00:08<00:00, 106MiB/s]


CLIP text encoder ready.


In [15]:
emotion_prompts = [f"a photo that conveys {e}" for e in EMOTIONS]
emotion_tokens  = clip.tokenize(emotion_prompts).to(DEVICE)

with torch.no_grad():
    emotion_text_features = clip_model.encode_text(emotion_tokens)
    emotion_text_features = emotion_text_features / emotion_text_features.norm(dim=-1, keepdim=True)

print(f"Emotion text features: {emotion_text_features.shape}")

Emotion text features: torch.Size([6, 768])


In [16]:
def score_emotions_batch(clip_feats: np.ndarray) -> np.ndarray:
    """Compute softmax similarity scores for all 6 emotions.
    clip_feats: (B, 768) float32 numpy array (already L2-normalised)
    Returns: (B, 6) float32 numpy array
    """
    feats = torch.tensor(clip_feats, device=DEVICE).to(emotion_text_features.dtype)
    with torch.no_grad():
        sims = (feats @ emotion_text_features.T).softmax(dim=-1)
    return sims.cpu().float().numpy()


all_scores = []
for i in range(0, len(clip_matrix), BATCH_SIZE):
    batch = clip_matrix[i : i + BATCH_SIZE]
    all_scores.append(score_emotions_batch(batch))

emotion_scores_matrix = np.concatenate(all_scores, axis=0)  # (N, 6)
print(f"Emotion scores computed: {emotion_scores_matrix.shape}")

Emotion scores computed: (16815, 6)


In [17]:
keep_cols = [col for col in ["name", "local_sudo hostnamectl set-hostname path", "dataset", "memscore", "original_emotion", "error"]
             if col in base_df.columns]
result_df = base_df[keep_cols].copy()

for i, e in enumerate(EMOTIONS):
    result_df[f"emotion_{e}"] = emotion_scores_matrix[:, i]

result_df["clip_emotion"]      = [EMOTIONS[i] for i in emotion_scores_matrix.argmax(axis=1)]
result_df["clip_emotion_conf"] = emotion_scores_matrix.max(axis=1)

print(result_df["clip_emotion"].value_counts())
result_df.head(3)

clip_emotion
anger        11394
sadness       2304
fear          1136
surprise      1038
disgust        487
happiness      456
Name: count, dtype: int64


,name,dataset,memscore,original_emotion,error,emotion_happiness,emotion_sadness,emotion_fear,emotion_anger,emotion_disgust,emotion_surprise,clip_emotion,clip_emotion_conf
0,original/2.jpg,abnormal,0.694118,,False,0.165649,0.167725,0.165283,0.167358,0.166138,0.167725,sadness,0.167725
1,original/4.jpg,art,0.792208,amusement,False,0.164673,0.166748,0.166016,0.169067,0.166382,0.167114,anger,0.169067
2,original/30.jpg,abnormal,0.848101,,False,0.165039,0.167480,0.167114,0.171387,0.164673,0.164429,anger,0.171387


In [18]:
emo_df = result_df[result_df.original_emotion != '']
emo_df.shape

(540, 13)

In [19]:
y_true = emo_df["original_emotion"].str.lower().str.strip()
y_pred = emo_df["clip_emotion"].str.lower().str.strip()


In [20]:
common_labels = sorted(set(list(y_true.unique()) + EMOTIONS))
common_labels

['amusement',
 'anger',
 'awe',
 'contentment',
 'disgust',
 'excitement',
 'fear',
 'happiness',
 'sad',
 'sadness',
 'surprise']

In [21]:
list(y_true.unique()), list(y_pred.unique())

(['amusement', 'excitement', 'awe', 'contentment', 'sad'],
 ['anger', 'happiness', 'sadness', 'disgust', 'surprise', 'fear'])

In [22]:
result_df.clip_emotion.value_counts()

clip_emotion
anger        11394
sadness       2304
fear          1136
surprise      1038
disgust        487
happiness      456
Name: count, dtype: int64

In [23]:
emo_df.groupby("original_emotion")["clip_emotion"].value_counts()

original_emotion  clip_emotion
amusement         anger            29
                  happiness        24
                  surprise         22
                  sadness          18
                  fear              5
                  disgust           2
awe               sadness          59
                  anger            33
                  fear              6
                  surprise          3
                  happiness         1
contentment       sadness          36
                  anger            30
                  surprise          1
excitement        anger            33
                  sadness          29
                  happiness        28
                  surprise         10
                  fear              5
sad               sadness         140
                  fear             13
                  anger            12
                  surprise          1
Name: count, dtype: int64

## 3 — Generate image captions with BLIP-2

For each image we produce a short natural-language description.

In [25]:
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
blip_model.eval()
print("BLIP-2 ready.")

Loading weights: 100%|██████████| 1247/1247 [00:00<00:00, 1550.07it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 7.64 GiB of which 33.19 MiB is free. Including non-PyTorch memory, this process has 6.11 GiB memory in use. Process 27426 has 878.00 MiB memory in use. Of the allocated memory 5.66 GiB is allocated by PyTorch, and 273.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
def get_caption(image: Image.Image) -> str:
    """Generate a descriptive caption for a single PIL image."""
    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    ).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)

    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=60)

    return blip_processor.decode(out[0], skip_special_tokens=True).strip()

In [ ]:
caption_cache_path = OUTPUT_DIR / "captions_cache.parquet"

if caption_cache_path.exists():
    caption_cache = pd.read_parquet(caption_cache_path).set_index("name")["caption"].to_dict()
    print(f"Resuming: {len(caption_cache)} captions already in cache.")
else:
    caption_cache = {}
    print("Starting caption generation from scratch.")

names_todo = [n for n in result_df["name"].tolist() if n not in caption_cache]
print(f"Images remaining: {len(names_todo)}")

Resuming: 5500 captions already in cache.
Images remaining: 11315


In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
from pathlib import Path

SAVE_EVERY = 500

caption_cache_path = Path(caption_cache_path)

if caption_cache_path.exists():
    cache_df = pd.read_parquet(caption_cache_path)
    caption_cache = dict(zip(cache_df["name"], cache_df["caption"]))
    print(f"Loaded existing cache with {len(caption_cache)} entries")
else:
    caption_cache = {}

names_remaining = [n for n in names_todo if n not in caption_cache]
print(f"Processing {len(names_remaining)} remaining images")

for idx, name in enumerate(tqdm(names_remaining)):
    local_path = IMAGES_DIR / name

    try:
        image = Image.open(local_path).convert("RGB")
        caption = get_caption(image)
    except Exception:
        caption = ""

    caption_cache[name] = caption


    if (idx + 1) % SAVE_EVERY == 0:
        temp_path = caption_cache_path.with_suffix(".tmp")

        cache_df = pd.DataFrame({
            "name": list(caption_cache.keys()),
            "caption": list(caption_cache.values())
        })

        cache_df.to_parquet(temp_path, index=False)


        temp_path.replace(caption_cache_path)

        print(f"Cache saved ({len(caption_cache)} captions)")

temp_path = caption_cache_path.with_suffix(".tmp")

cache_df = pd.DataFrame({
    "name": list(caption_cache.keys()),
    "caption": list(caption_cache.values())
})

cache_df.to_parquet(temp_path, index=False)
temp_path.replace(caption_cache_path)

print(f"Caption generation complete. Total: {len(caption_cache)}")

Loaded existing cache with 7500 entries
Processing 9315 remaining images


  5%|▌         | 500/9315 [02:54<54:33,  2.69it/s]

Cache saved (8000 captions)


 11%|█         | 1000/9315 [12:13<2:39:00,  1.15s/it]

Cache saved (8500 captions)


 16%|█▌        | 1500/9315 [21:35<2:21:23,  1.09s/it]

Cache saved (9000 captions)


 17%|█▋        | 1623/9315 [23:52<2:20:49,  1.10s/it]

In [ ]:
# SAVE_EVERY = 500   # flush cache to disk every N captions

# for idx, name in enumerate(tqdm(names_todo)):
#     local_path = IMAGES_DIR / name
#     try:2
#         image = Image.open(local_path).convert("RGB")
#         caption = get_caption(image)
#     except Exception:
#         caption = ""  # keep empty on error; error flag already in base_df

#     caption_cache[name] = caption

#     if (idx + 1) % SAVE_EVERY == 0:
#         cache_df = pd.DataFrame({"name": list(caption_cache.keys()),
#                                   "caption": list(caption_cache.values())})
#         cache_df.to_parquet(caption_cache_path, index=False)
#         print(f"  Cache saved ({len(caption_cache)} captions)")

# # Final flush
# cache_df = pd.DataFrame({"name": list(caption_cache.keys()),
#                           "caption": list(caption_cache.values())})
# cache_df.to_parquet(caption_cache_path, index=False)
# print(f"Caption generation complete. Total: {len(caption_cache)}")

In [ ]:
# Merge captions into result_df
result_df["caption"] = result_df["name"].map(caption_cache).fillna("")
print(f"Captions attached. Non-empty: {(result_df['caption'] != '').sum()}")

## 4 — SBERT caption embeddings

In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim

captions = result_df["caption"].fillna("").tolist()
caption_matrix = sbert_model.encode(
    captions,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)  # (N, 384)

print(f"SBERT embeddings shape: {caption_matrix.shape}")

## 5 — Save outputs

In [ ]:
# Parquet with full metadata
parquet_out = OUTPUT_DIR / "lamem_features_v2_full.parquet"
result_df.to_parquet(parquet_out, index=False)
print(f"Parquet saved -> {parquet_out}  shape: {result_df.shape}")

# CLIP embeddings (same data, copied to new output dir for self-contained pipeline)
clip_out = OUTPUT_DIR / "clip_embeddings.npy"
np.save(clip_out, clip_matrix)
print(f"CLIP embeddings -> {clip_out}  shape: {clip_matrix.shape}")

# SBERT caption embeddings
caption_emb_out = OUTPUT_DIR / "caption_embeddings.npy"
np.save(caption_emb_out, caption_matrix)
print(f"SBERT embeddings -> {caption_emb_out}  shape: {caption_matrix.shape}")

## 6 — Sanity check

In [ ]:
print("=" * 50)
print(f"lamem_features_v2_full.parquet  : {result_df.shape}")
print(f"clip_embeddings.npy             : {clip_matrix.shape}")
print(f"caption_embeddings.npy          : {caption_matrix.shape}")
print()
print("Emotion distribution (new labels):")
print(result_df["clip_emotion"].value_counts())
print()
print("Sample rows:")
result_df[["name", "memscore", "clip_emotion", "clip_emotion_conf", "caption"]].head(5)